# Python Advanced - Day 8 Homework

**Runbook alignment (Day 8, Hours 29-32):**
- Hour 29: Deeper SQL practice (graded) with ORM as an optional extension
- Hour 30: Integrate DB into the app (service layer uses repository)
- Hour 31: Search or filter + pagination patterns (practical)
- Hour 32: Checkpoint 4: Data-driven app milestone

This homework makes the app truly data-driven by deepening database skills, wiring the repository into the service layer, and adding search patterns.

For grading consistency, complete the deeper SQL practice outputs for Hour 29; the ORM angle is an optional extension note, not an alternate graded path.



### Autograder Integration Requirements

This assignment is graded automatically by the **Global Autograder**.

- All solution code must live in this notebook and run without errors.
- The autograder converts the submission notebook to `advanced_day8.py` and grades it against `criteria.json`.
- Required outputs must exactly match the labeled canonical strings in `criteria.json`.
- Keep output deterministic: no randomness, live timestamps, network calls, or hidden external state.
- Do not rename this notebook or move the config files (`criteria.json`, `setup.json`, `feedback.json`).
- If you stage work in `submissions/`, keep exactly one real `*Advanced_Day8*.ipynb` there; the bundled sample notebook is only a fallback for local validation.
- Keep file access inside the assignment directory only.
- Run all cells before submitting.



## Part 1: Hour 29 - Deeper SQL practice (required) with ORM as an optional extension

### Goal
Build the deeper SQL query set and mention the ORM mapping idea as an optional extension note.

### Starter
- Start from the scaffold in the next code cell and keep names aligned with the prompts.
- Use this implementation direction: Deliver the deeper SQL path first, then capture ORM as an optional extension note if you explored it.

### Steps
- Implement the requested behavior for this hour in the provided scaffold.
- Run the notebook cells and verify the required labeled outputs appear exactly as specified.
- Refactor only after outputs are stable and deterministic.

### Success Criteria
Use the Definition of Done below (from issue #483) as the authoritative grading contract for this part.

#### Definition of Done (Part 1)
- Convert this notebook to `advanced_day8.py`, then run `python advanced_day8.py` with no runtime errors.
- Satisfy every aligned grading check:
  - **Hour 29 - Optional ORM slice (SQLAlchemy) OR deeper SQL practice**
    - Command: `python advanced_day8.py`
    - Expected outcomes (exact markers):
      - `Hour 29 | chosen path: deeper SQL practice`
      - `Hour 29 | filtered open tasks: 2`
      - `Hour 29 | aggregate done count: 1`
      - `Hour 29 | sorted query first title: Archive logs`
      - `Hour 29 | extension note: orm maps rows to objects`
- Keep required output deterministic (labels, capitalization, punctuation, and ordering).

### Common Failure Modes
- Adding ORM complexity without understanding the underlying query shape hides bugs.
- Changing required marker text (labels, capitalization, punctuation, or ordering).
- Relying on non-deterministic behavior (timestamps, random order, external state, or unstable environment values).


In [ ]:
# Hour 29 scaffold: deeper SQL queries with deterministic seed data
import sqlite3
from pathlib import Path


def reset_day8_db(db_path: Path) -> sqlite3.Connection:
    db_path.parent.mkdir(parents=True, exist_ok=True)
    conn = sqlite3.connect(db_path)
    cur = conn.cursor()
    cur.execute("DROP TABLE IF EXISTS tasks")
    cur.execute(
        """
        CREATE TABLE tasks (
            task_id TEXT PRIMARY KEY,
            title TEXT NOT NULL,
            status TEXT NOT NULL,
            done INTEGER NOT NULL
        )
        """
    )
    cur.executemany(
        "INSERT INTO tasks(task_id, title, status, done) VALUES (?, ?, ?, ?)",
        [
            ("task-100", "Archive logs", "open", 0),
            ("task-101", "Prepare sprint report", "open", 0),
            ("task-102", "Close old tickets", "done", 1),
        ],
    )
    conn.commit()
    return conn


assignment_root = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
db_file = assignment_root / "data" / "tracker.db"
conn = reset_day8_db(db_file)
cur = conn.cursor()

open_count = cur.execute("SELECT COUNT(*) FROM tasks WHERE status = 'open'").fetchone()[0]
done_count = cur.execute("SELECT COALESCE(SUM(done), 0) FROM tasks").fetchone()[0]
first_title = cur.execute("SELECT title FROM tasks ORDER BY title ASC LIMIT 1").fetchone()[0]

print("Hour 29 | chosen path: deeper SQL practice")
print(f"Hour 29 | filtered open tasks: {open_count}")
print(f"Hour 29 | aggregate done count: {done_count}")
print(f"Hour 29 | sorted query first title: {first_title}")
print("Hour 29 | extension note: orm maps rows to objects")


### Practice and Review

- Explain why this hour matters to the capstone.
- Note how you would test or demo this hour's deliverable.
- Compare your labels to the canonical contract before submission.



## Part 2: Hour 30 - Integrate DB into the app (service layer uses repository)

### Goal
Build a service layer that uses the repository as its data source.

### Starter
- Start from the scaffold in the next code cell and keep names aligned with the prompts.
- Use this implementation direction: Wire the repository under the service layer rather than exposing DB code to the UI.

### Steps
- Implement the requested behavior for this hour in the provided scaffold.
- Run the notebook cells and verify the required labeled outputs appear exactly as specified.
- Refactor only after outputs are stable and deterministic.

### Success Criteria
Use the Definition of Done below (from issue #483) as the authoritative grading contract for this part.

#### Definition of Done (Part 2)
- Convert this notebook to `advanced_day8.py`, then run `python advanced_day8.py` with no runtime errors.
- Satisfy every aligned grading check:
  - **Hour 30 - Integrate DB into the app (service layer uses repository)**
    - Command: `python advanced_day8.py`
    - Expected outcomes (exact markers):
      - `Hour 30 | repository plugged in: yes`
      - `Hour 30 | service load count: 3`
      - `Hour 30 | add via service: stored in db`
      - `Hour 30 | list via service: db-backed`
      - `Hour 30 | integration rule: ui talks to service, service talks to repository`
- Keep required output deterministic (labels, capitalization, punctuation, and ordering).

### Common Failure Modes
- Letting the UI call sqlite directly breaks separation of concerns.
- Changing required marker text (labels, capitalization, punctuation, or ordering).
- Relying on non-deterministic behavior (timestamps, random order, external state, or unstable environment values).


In [ ]:
# Hour 30 scaffold: repository + service integration over sqlite
from dataclasses import dataclass


@dataclass
class DbTask:
    task_id: str
    title: str
    status: str


class TaskRepository:
    def __init__(self, connection: sqlite3.Connection) -> None:
        self._conn = connection

    def list_tasks(self) -> list[DbTask]:
        rows = self._conn.execute(
            "SELECT task_id, title, status FROM tasks ORDER BY task_id"
        ).fetchall()
        return [DbTask(*row) for row in rows]

    def add_task(self, task: DbTask) -> None:
        self._conn.execute(
            "INSERT INTO tasks(task_id, title, status, done) VALUES (?, ?, ?, ?)",
            (task.task_id, task.title, task.status, 1 if task.status == "done" else 0),
        )
        self._conn.commit()


class TaskService:
    def __init__(self, repository: TaskRepository) -> None:
        self._repository = repository

    def load(self) -> list[DbTask]:
        return self._repository.list_tasks()

    def add(self, task: DbTask) -> None:
        self._repository.add_task(task)


repo = TaskRepository(conn)
service = TaskService(repo)
loaded = service.load()
service.add(DbTask(task_id="task-103", title="Publish report", status="open"))

print("Hour 30 | repository plugged in: yes")
print(f"Hour 30 | service load count: {len(loaded)}")
print("Hour 30 | add via service: stored in db")
print("Hour 30 | list via service: db-backed")
print("Hour 30 | integration rule: ui talks to service, service talks to repository")


### Practice and Review

- Explain why this hour matters to the capstone.
- Note how you would test or demo this hour's deliverable.
- Compare your labels to the canonical contract before submission.



## Part 3: Hour 31 - Search or filter + pagination patterns (practical)

### Goal
Build search and pagination logic with stable ordering.

### Starter
- Start from the scaffold in the next code cell and keep names aligned with the prompts.
- Use this implementation direction: Apply stable sorting before slicing pages so navigation stays predictable.

### Steps
- Implement the requested behavior for this hour in the provided scaffold.
- Run the notebook cells and verify the required labeled outputs appear exactly as specified.
- Refactor only after outputs are stable and deterministic.

### Success Criteria
Use the Definition of Done below (from issue #483) as the authoritative grading contract for this part.

#### Definition of Done (Part 3)
- Convert this notebook to `advanced_day8.py`, then run `python advanced_day8.py` with no runtime errors.
- Satisfy every aligned grading check:
  - **Hour 31 - Search or filter + pagination patterns (practical)**
    - Command: `python advanced_day8.py`
    - Expected outcomes (exact markers):
      - `Hour 31 | search term: report`
      - `Hour 31 | filtered count: 2`
      - `Hour 31 | page size: 2`
      - `Hour 31 | page 2 ids: task-103, task-104`
      - `Hour 31 | query rule: sort before paginating`
- Keep required output deterministic (labels, capitalization, punctuation, and ordering).

### Common Failure Modes
- Paginating unsorted results causes records to jump between pages.
- Changing required marker text (labels, capitalization, punctuation, or ordering).
- Relying on non-deterministic behavior (timestamps, random order, external state, or unstable environment values).


In [ ]:
# Hour 31 scaffold: stable filter + pagination logic
search_records = [
    {"task_id": "task-101", "title": "Prepare sprint report"},
    {"task_id": "task-102", "title": "Review weekly notes"},
    {"task_id": "task-103", "title": "Archive logs"},
    {"task_id": "task-104", "title": "Publish report"},
]

search_term = "report"
filtered = [item for item in search_records if search_term in item["title"].lower()]
ordered = sorted(search_records, key=lambda item: item["task_id"])
page_size = 2
page_number = 2
start = (page_number - 1) * page_size
end = start + page_size
page_two = ordered[start:end]
page_two_ids = ", ".join(item["task_id"] for item in page_two)

print(f"Hour 31 | search term: {search_term}")
print(f"Hour 31 | filtered count: {len(filtered)}")
print(f"Hour 31 | page size: {page_size}")
print(f"Hour 31 | page 2 ids: {page_two_ids}")
print("Hour 31 | query rule: sort before paginating")


### Practice and Review

- Explain why this hour matters to the capstone.
- Note how you would test or demo this hour's deliverable.
- Compare your labels to the canonical contract before submission.



## Part 4: Hour 32 - Checkpoint 4: Data-driven app milestone

### Goal
Build a checkpoint build where the app is genuinely database-backed.

### Starter
- Start from the scaffold in the next code cell and keep names aligned with the prompts.
- Use this implementation direction: Demonstrate persistence-backed flows end to end at the checkpoint.

### Steps
- Implement the requested behavior for this hour in the provided scaffold.
- Run the notebook cells and verify the required labeled outputs appear exactly as specified.
- Refactor only after outputs are stable and deterministic.

### Success Criteria
Use the Definition of Done below (from issue #483) as the authoritative grading contract for this part.

#### Definition of Done (Part 4)
- Convert this notebook to `advanced_day8.py`, then run `python advanced_day8.py` with no runtime errors.
- Satisfy every aligned grading check:
  - **Hour 32 - Checkpoint 4: Data-driven app milestone**
    - Command: `python advanced_day8.py`
    - Expected outcomes (exact markers):
      - `Hour 32 | db path: data/tracker.db`
      - `Hour 32 | add flow: persisted`
      - `Hour 32 | search flow: persisted data returned`
      - `Hour 32 | edit flow: persisted`
      - `Hour 32 | checkpoint result: data-driven app milestone`
- Keep required output deterministic (labels, capitalization, punctuation, and ordering).

### Common Failure Modes
- Calling it data-driven without proving reads and writes against the database is premature.
- Changing required marker text (labels, capitalization, punctuation, or ordering).
- Relying on non-deterministic behavior (timestamps, random order, external state, or unstable environment values).


In [ ]:
# Hour 32 scaffold: checkpoint flow proving persistence-backed behavior
service.add(DbTask(task_id="task-104", title="Run smoke tests", status="open"))

persisted_before = service.load()
search_hits = [task for task in persisted_before if "report" in task.title.lower()]

conn.execute(
    "UPDATE tasks SET title = ? WHERE task_id = ?",
    ("Archive logs", "task-100"),
)
conn.commit()
edited_title = conn.execute(
    "SELECT title FROM tasks WHERE task_id = ?",
    ("task-100",),
).fetchone()[0]

print(f"Hour 32 | db path: {db_file.as_posix()}")
print("Hour 32 | add flow: persisted")
print("Hour 32 | search flow: persisted data returned")
print("Hour 32 | edit flow: persisted")
print("Hour 32 | checkpoint result: data-driven app milestone")

conn.close()


### Practice and Review

- Explain why this hour matters to the capstone.
- Note how you would test or demo this hour's deliverable.
- Compare your labels to the canonical contract before submission.


## Reflection Questions

1. Which hour felt strongest today, and why?
2. Which output label was hardest to make deterministic?
3. What would you improve before a checkpoint or demo?

### Your Answers


## Submission Checklist

- My completed notebook stays named `Advanced_Day8_homework.ipynb`.
- I copied a finished submission notebook into `submissions/` before grading.
- My output labels match `criteria.json` exactly.
- I avoided randomness, live timestamps, and hidden external state.
- I ran all cells and fixed errors before submission.


## Authenticity Guardrail

Use the grading contract in `criteria.json` to verify required labels, but keep the implementation real:

- Use SQL queries and repository/service code paths (not print-only shortcuts).
- Keep deterministic seed data so your outputs remain stable.
- Demonstrate each checkpoint flow through actual reads/writes before printing status lines.
